<a href="https://colab.research.google.com/github/Abhyuday-19/Energy-optimizer-agent/blob/main/Energy_Consumption_Optimizer_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu

import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

import warnings
warnings.filterwarnings("ignore")

print("Setup done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.1 MB/s eta 0:00:00
Setup done


In [2]:
appliance_wattage = {
    "Air Conditioner (1.5 Ton, 3 Star)": 1500,
    "Air Conditioner (1.5 Ton, 5 Star)": 1150,
    "Air Conditioner (1 Ton, Inverter)": 900,
    "Ceiling Fan": 75,
    "Table Fan": 55,
    "Cooler (Desert)": 230,
    "Refrigerator (Double Door, 250L)": 150,
    "Refrigerator (Single Door, 180L)": 100,
    "Washing Machine (Front Load)": 2000,
    "Washing Machine (Top Load)": 500,
    "Geyser / Water Heater (15L)": 2000,
    "Geyser / Water Heater (25L)": 2500,
    "LED TV (43 inch)": 70,
    "LED TV (55 inch)": 110,
    "Microwave Oven": 1200,
    "Induction Cooktop": 2000,
    "Electric Iron": 1000,
    "Laptop": 65,
    "Desktop Computer": 200,
    "Wi-Fi Router": 10,
    "Tubelight / LED Bulb (per unit)": 15,
    "Mixer Grinder": 500,
    "Water Pump (0.5 HP)": 370,
    "Chimney (Kitchen)": 250,
    "Electric Kettle": 1500,
    "Inverter (Home, charging load)": 300
}

energy_tips = [
    {"id": 1, "category": "AC", "tip": "AC ka temperature 24-26°C rakhein, har 1°C kam karne se 6-8% zyada bijli khti hai."},
    {"id": 2, "category": "AC", "tip": "AC ke saath ceiling fan chalayein, isse aap temperature 2-3 degree zyada rakh sakte hain aur comfort same rahega."},
    {"id": 3, "category": "AC", "tip": "5-star rated inverter AC, 3-star fixed-speed AC se 20-30% kam bijli kharch karta hai."},
    {"id": 4, "category": "AC", "tip": "AC filter har 2-3 hafte mein clean karein, dirty filter se compressor zyada mehnat karta hai aur bijli zyada lagti hai."},
    {"id": 5, "category": "AC", "tip": "AC chalane se pehle room ke curtains/blinds band kar dein dopahar mein, direct sunlight se room garam hota hai."},
    {"id": 6, "category": "AC", "tip": "Sleep mode use karein raat ko, ye automatically temperature thoda badha deta hai jab aap so jaate hain."},
    {"id": 7, "category": "AC", "tip": "AC ko bar-bar on-off na karein, isse compressor pe extra load padta hai jo zyada bijli kharch karta hai."},
    {"id": 8, "category": "Refrigerator", "tip": "Fridge ko deewar se kam se kam 4-6 inch door rakhein taaki heat properly dissipate ho sake."},
    {"id": 9, "category": "Refrigerator", "tip": "Fridge ka door bar-bar na kholein aur zyada der tak khula na rakhein, andar ka temperature maintain karne mein extra bijli lagti hai."},
    {"id": 10, "category": "Refrigerator", "tip": "Garam khana fridge mein rakhne se pehle room temperature tak thanda hone dein."},
    {"id": 11, "category": "Refrigerator", "tip": "Fridge ki temperature setting 4°C (fridge) aur -18°C (freezer) se zyada low na rakhein, zaroorat se zyada thanda karna bijli waste karta hai."},
    {"id": 12, "category": "Washing Machine", "tip": "Full load mein hi washing machine chalayein, half load mein bhi utni hi bijli lagti hai."},
    {"id": 13, "category": "Washing Machine", "tip": "Cold water wash cycle use karein jab possible ho, pani garam karna sabse zyada bijli kharch karta hai washing machine mein."},
    {"id": 14, "category": "Geyser", "tip": "Geyser ko zaroorat se 15-20 min pehle hi on karein, continuously on rakhne se bahut bijli waste hoti hai."},
    {"id": 15, "category": "Geyser", "tip": "Geyser ka temperature 50-55°C rakhein, isse zyada garam karna extra bijli lagti hai bina extra benefit ke."},
    {"id": 16, "category": "Geyser", "tip": "Instant geyser (chhoti capacity) storage geyser se kam bijli kharch karta hai agar usage kam hai to."},
    {"id": 17, "category": "Lighting", "tip": "LED bulbs use karein, CFL se 50% aur incandescent se 80% kam bijli kharch karte hain same brightness ke liye."},
    {"id": 18, "category": "Lighting", "tip": "Din mein natural light use karein, khidkiyan khuli rakhein jab tak possible ho."},
    {"id": 19, "category": "Lighting", "tip": "Jis room mein koi na ho wahan ki lights band kar dein, chhoti aadat bade savings deti hai."},
    {"id": 20, "category": "General", "tip": "TV, set-top box, charger jaise devices standby mode mein bhi bijli kharch karte hain, poora switch off karein."},
    {"id": 21, "category": "General", "tip": "Smart power strip use karein jisse ek click mein saare standby devices band ho jayein."},
    {"id": 22, "category": "General", "tip": "5-star BEE rated appliances kharidein, shuru mein thoda mehenge hote hain par 2-3 saal mein bijli bill se recover ho jaata hai."},
    {"id": 23, "category": "General", "tip": "Appliances ki regular servicing karayein, purane/dirty appliances zyada bijli kharch karte hain."},
    {"id": 24, "category": "General", "tip": "Solar water heater install karne se geyser ka 60-70% load khatam ho sakta hai, especially North India mein."},
    {"id": 25, "category": "Kitchen", "tip": "Induction cooktop gas se bijli mein zyada efficient hota hai agar flat-bottom bartan use karein."},
    {"id": 26, "category": "Kitchen", "tip": "Microwave, oven se kam bijli aur kam time leta hai chhoti quantity garam karne mein."},
    {"id": 27, "category": "Kitchen", "tip": "Mixer grinder continuously 2-3 min se zyada na chalayein, motor garam hoke zyada current khenchta hai."},
    {"id": 28, "category": "Billing", "tip": "Electricity bill slab-based hota hai — agar aap next slab ke close hain, thoda sa usage kam karke aap poore bill mein bade % savings pa sakte hain."},
    {"id": 29, "category": "Billing", "tip": "Peak hours (sham 6-10 baje) mein heavy appliances (washing machine, iron) avoid karein, kuch states mein time-of-day pricing hoti hai."},
    {"id": 30, "category": "Billing", "tip": "Monthly meter reading note karte rahein taaki aapko pata chale kis mahine consumption achanak badha."},
    {"id": 31, "category": "Seasonal-Summer", "tip": "Garmi mein cooler, AC se 70-80% kam bijli kharch karta hai agar humidity kam ho to."},
    {"id": 32, "category": "Seasonal-Summer", "tip": "Terrace/roof pe halka paint ya white sheet se ghar thanda rehta hai, AC ka load kam hota hai."},
    {"id": 33, "category": "Seasonal-Summer", "tip": "Exhaust fan subah-shaam use karein garam hawa bahar nikalne ke liye, AC kam chalana padega."},
    {"id": 34, "category": "Seasonal-Winter", "tip": "Geyser ka usage winter mein sabse zyada badhta hai, time-based usage (sirf nahane se pehle) savings deta hai."},
    {"id": 35, "category": "Seasonal-Winter", "tip": "Room heater ki jagah agar possible ho to layered clothing use karein, heater bahut zyada bijli kharch karta hai (1500-2500W)."},
    {"id": 36, "category": "Seasonal-Festival", "tip": "Diwali/festival lights LED string lights use karein, traditional bulbs se 80% kam bijli kharch karte hain."},
    {"id": 37, "category": "Seasonal-Festival", "tip": "Festival decoration lights ke liye timer laga dein, raat 11-12 baje ke baad automatically band ho jayein."},
    {"id": 38, "category": "Water Pump", "tip": "Water pump ko overhead tank full hote hi band karein, automatic water level sensor laga sakte hain."},
    {"id": 39, "category": "Water Pump", "tip": "Pump ki pipeline mein leakage check karein, leakage se pump zyada der chalana padta hai."},
    {"id": 40, "category": "Solar", "tip": "Rooftop solar panel lagwane se grid se bijli consumption 40-90% tak kam ho sakta hai, government subsidy bhi milti hai."},
    {"id": 41, "category": "Solar", "tip": "Net metering connection lein agar solar panel lagate hain, extra bijli grid ko bech sakte hain."},
    {"id": 42, "category": "Behavioral", "tip": "Family mein sabko bijli bill dikhayein aur ek simple goal set karein (jaise '10% kam karna is mahine'), awareness se bhi savings hoti hai."},
    {"id": 43, "category": "Behavioral", "tip": "Appliance use karne ka ek schedule banayein especially high-wattage appliances ke liye taaki overlap na ho."},
    {"id": 44, "category": "Behavioral", "tip": "Bijli bill ko sirf paisa nahi, carbon footprint ke roop mein bhi dekhein — kam usage environment ke liye bhi acha hai."},
    {"id": 45, "category": "High-Load", "tip": "Geyser, AC, aur iron ek saath kabhi na chalayein agar connection load limited hai, isse MCB trip bhi ho sakta hai aur peak load bhi badhta hai."},
    {"id": 46, "category": "High-Load", "tip": "Iron karte waqt saare kapde ek saath rakhein, baar-baar heat-up karna (on-off) zyada bijli kharch karta hai."},
    {"id": 47, "category": "High-Load", "tip": "Induction cooktop ka use bade bartan ke liye karein, chhote bartan se heat zyada waste hoti hai."},
    {"id": 48, "category": "Monitoring", "tip": "Smart plug ya energy monitor use karke pata karein kaunsa appliance sabse zyada bijli kharch kar raha hai."},
    {"id": 49, "category": "Monitoring", "tip": "Monthly usage pattern track karein — achanak spike dikhe to kisi appliance mein fault ho sakta hai."},
    {"id": 50, "category": "Monitoring", "tip": "BEE star rating label har appliance pe check karein kharidte waqt, zyada stars = kam running cost."}
]

import json
with open("appliance_wattage.json", "w") as f:
    json.dump(appliance_wattage, f, indent=2)
with open("energy_tips.json", "w") as f:
    json.dump(energy_tips, f, indent=2)

print("Appliances:", len(appliance_wattage))
print("Tips:", len(energy_tips))

Appliances: 26
Tips: 50


In [3]:
# PSPCL (Punjab) domestic tariff — approximate, PSERC FY 2026-27 tariff order
PUNJAB_SLABS = [
    (100, 3.85),
    (300, 5.25),
    (500, 6.95),
    (float('inf'), 8.15)
]
FIXED_CHARGE = 130
ELECTRICITY_DUTY = 0.05
FREE_UNITS = 300   # Punjab govt scheme: pehle 300 units/month FREE, cross kiya to POORA bill charge


def calculate_slab_bill(units):
    if units <= 0:
        return 0.0
    remaining = units
    prev_limit = 0
    total = 0.0
    for limit, rate in PUNJAB_SLABS:
        slab_units = min(remaining, limit - prev_limit)
        if slab_units <= 0:
            break
        total += slab_units * rate
        remaining -= slab_units
        prev_limit = limit
        if remaining <= 0:
            break
    return total


def calculate_bill(units):
    if units <= FREE_UNITS:
        return {
            "units": units, "free_scheme_applied": True,
            "energy_charge": 0.0, "fixed_charge": 0.0, "duty": 0.0, "total": 0.0,
            "note": f"Within {FREE_UNITS} free units — bill is Rs 0 under Punjab's domestic scheme."
        }
    energy_charge = calculate_slab_bill(units)
    duty = energy_charge * ELECTRICITY_DUTY
    total = energy_charge + FIXED_CHARGE + duty
    return {
        "units": units, "free_scheme_applied": False,
        "energy_charge": round(energy_charge, 2), "fixed_charge": FIXED_CHARGE,
        "duty": round(duty, 2), "total": round(total, 2),
        "note": "Crossed 300 units — entire consumption billed at slab rates, not just the extra units."
    }


def units_to_next_slab_saving(units):
    if units <= FREE_UNITS:
        return None
    over = units - FREE_UNITS
    return {
        "units_over_free_limit": over,
        "advice": f"Aap free 300-unit limit se {over} units zyada use kar rahe ho. "
                  f"Agar is mahine {over} units kam kar lein, poora bill 0 ho sakta hai."
    }


# quick test
for u in [250, 300, 301, 500]:
    print(u, "units ->", calculate_bill(u))

250 units -> {'units': 250, 'free_scheme_applied': True, 'energy_charge': 0.0, 'fixed_charge': 0.0, 'duty': 0.0, 'total': 0.0, 'note': "Within 300 free units — bill is Rs 0 under Punjab's domestic scheme."}
300 units -> {'units': 300, 'free_scheme_applied': True, 'energy_charge': 0.0, 'fixed_charge': 0.0, 'duty': 0.0, 'total': 0.0, 'note': "Within 300 free units — bill is Rs 0 under Punjab's domestic scheme."}
301 units -> {'units': 301, 'free_scheme_applied': False, 'energy_charge': 1441.95, 'fixed_charge': 130, 'duty': 72.1, 'total': 1644.05, 'note': 'Crossed 300 units — entire consumption billed at slab rates, not just the extra units.'}
500 units -> {'units': 500, 'free_scheme_applied': False, 'energy_charge': 2825.0, 'fixed_charge': 130, 'duty': 141.25, 'total': 3096.25, 'note': 'Crossed 300 units — entire consumption billed at slab rates, not just the extra units.'}


In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Chhota, fast model — brochure mein "cost-effective, edge-deployable" wala point yahi hai
model = SentenceTransformer('all-MiniLM-L6-v2')

# ---- 1. Embed all tips ONCE and cache (optimization: dobara compute mat karo) ----
tip_texts = [t["tip"] for t in energy_tips]
tip_embeddings = model.encode(tip_texts, show_progress_bar=True, normalize_embeddings=True)
tip_embeddings = np.array(tip_embeddings, dtype="float32")

np.save("tip_embeddings.npy", tip_embeddings)  # cache — app restart pe dobara embed mat karo
print("Embeddings shape:", tip_embeddings.shape)

# ---- 2. FAISS index bana (normalized vectors + inner product = cosine similarity) ----
dimension = tip_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(tip_embeddings)
print("FAISS index mein tips:", index.ntotal)

# ---- 3. Retrieval function ----
def retrieve_tips(query, k=3):
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, k)
    results = []
    for score, i in zip(scores[0], idx[0]):
        results.append({**energy_tips[i], "relevance": round(float(score), 3)})
    return results

# ---- 4. Build a query from appliance usage (usage -> which appliances matter most) ----
def build_usage_query(usage_hours):
    """
    usage_hours: dict like {"Air Conditioner (1.5 Ton, 3 Star)": 8, "Refrigerator (Double Door, 250L)": 24}
                 (appliance name -> hours used per day)
    Returns the query text + ranked monthly kWh consumption per appliance.
    """
    consumption = []
    for appliance, hours in usage_hours.items():
        watt = appliance_wattage.get(appliance, 0)
        monthly_kwh = (watt * hours * 30) / 1000
        consumption.append((appliance, monthly_kwh))
    consumption.sort(key=lambda x: x[1], reverse=True)

    top_appliances = [a for a, _ in consumption[:3]]
    query = "Energy saving tips for: " + ", ".join(top_appliances)
    return query, consumption

# ---- 5. Test it ----
sample_usage = {
    "Air Conditioner (1.5 Ton, 3 Star)": 8,
    "Refrigerator (Double Door, 250L)": 24,
    "Geyser / Water Heater (15L)": 1,
    "LED TV (43 inch)": 4,
}

query, consumption = build_usage_query(sample_usage)
print("\nQuery:", query)
print("\nMonthly consumption (kWh), highest first:")
for appliance, kwh in consumption:
    print(f"  {appliance}: {kwh:.1f} kWh")

print("\nTop relevant tips:")
for tip in retrieve_tips(query, k=5):
    print(f"  [{tip['relevance']}] ({tip['category']}) {tip['tip']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (50, 384)
FAISS index mein tips: 50

Query: Energy saving tips for: Air Conditioner (1.5 Ton, 3 Star), Refrigerator (Double Door, 250L), Geyser / Water Heater (15L)

Monthly consumption (kWh), highest first:
  Air Conditioner (1.5 Ton, 3 Star): 360.0 kWh
  Refrigerator (Double Door, 250L): 108.0 kWh
  Geyser / Water Heater (15L): 60.0 kWh
  LED TV (43 inch): 8.4 kWh

Top relevant tips:
  [0.426] (Behavioral) Appliance use karne ka ek schedule banayein especially high-wattage appliances ke liye taaki overlap na ho.
  [0.409] (Monitoring) Smart plug ya energy monitor use karke pata karein kaunsa appliance sabse zyada bijli kharch kar raha hai.
  [0.397] (General) Solar water heater install karne se geyser ka 60-70% load khatam ho sakta hai, especially North India mein.
  [0.367] (General) Appliances ki regular servicing karayein, purane/dirty appliances zyada bijli kharch karte hain.
  [0.356] (Refrigerator) Fridge ko deewar se kam se kam 4-6 inch door rakhein taaki hea

In [5]:
def energy_agent(usage_hours, k_tips=5):
    """
    usage_hours: dict like {"Air Conditioner (1.5 Ton, 3 Star)": 8, "Refrigerator (Double Door, 250L)": 24}
                 (appliance name -> hours used PER DAY)
    Returns a full structured report: consumption breakdown, bill, proximity advice, and RAG tips.
    """
    # 1. Monthly consumption per appliance (kWh)
    consumption = []
    for appliance, hours in usage_hours.items():
        watt = appliance_wattage.get(appliance, 0)
        monthly_kwh = (watt * hours * 30) / 1000
        consumption.append({"appliance": appliance, "hours_per_day": hours, "monthly_kwh": round(monthly_kwh, 1)})
    consumption.sort(key=lambda x: x["monthly_kwh"], reverse=True)

    total_units = round(sum(c["monthly_kwh"] for c in consumption), 1)

    # 2. Billing (Punjab slab + free-scheme logic from Cell 3)
    bill = calculate_bill(total_units)
    proximity_advice = units_to_next_slab_saving(total_units)

    # 3. RAG retrieval (semantic search from Cell 4)
    top_appliances = [c["appliance"] for c in consumption[:3]]
    query = "Energy saving tips for: " + ", ".join(top_appliances)
    tips = retrieve_tips(query, k=k_tips)

    return {
        "consumption_breakdown": consumption,
        "total_monthly_units": total_units,
        "bill": bill,
        "proximity_advice": proximity_advice,
        "recommended_tips": tips,
        "top_consuming_appliances": top_appliances,
    }


# ---- Test it end-to-end ----
sample_usage = {
    "Air Conditioner (1.5 Ton, 3 Star)": 8,
    "Refrigerator (Double Door, 250L)": 24,
    "Geyser / Water Heater (15L)": 1,
    "LED TV (43 inch)": 4,
    "Ceiling Fan": 10,
}

result = energy_agent(sample_usage)

print("=" * 50)
print("CONSUMPTION BREAKDOWN (highest first)")
print("=" * 50)
for c in result["consumption_breakdown"]:
    print(f"  {c['appliance']}: {c['monthly_kwh']} kWh/month")

print(f"\nTotal monthly units: {result['total_monthly_units']} kWh")

print("\n" + "=" * 50)
print("BILL")
print("=" * 50)
print(result["bill"])

if result["proximity_advice"]:
    print("\n" + "=" * 50)
    print("PROXIMITY ADVICE")
    print("=" * 50)
    print(result["proximity_advice"]["advice"])

print("\n" + "=" * 50)
print("RECOMMENDED TIPS")
print("=" * 50)
for tip in result["recommended_tips"]:
    print(f"  [{tip['relevance']}] ({tip['category']}) {tip['tip']}")

CONSUMPTION BREAKDOWN (highest first)
  Air Conditioner (1.5 Ton, 3 Star): 360.0 kWh/month
  Refrigerator (Double Door, 250L): 108.0 kWh/month
  Geyser / Water Heater (15L): 60.0 kWh/month
  Ceiling Fan: 22.5 kWh/month
  LED TV (43 inch): 8.4 kWh/month

Total monthly units: 558.9 kWh

BILL
{'units': 558.9, 'free_scheme_applied': False, 'energy_charge': 3305.03, 'fixed_charge': 130, 'duty': 165.25, 'total': 3600.29, 'note': 'Crossed 300 units — entire consumption billed at slab rates, not just the extra units.'}

PROXIMITY ADVICE
Aap free 300-unit limit se 258.9 units zyada use kar rahe ho. Agar is mahine 258.9 units kam kar lein, poora bill 0 ho sakta hai.

RECOMMENDED TIPS
  [0.426] (Behavioral) Appliance use karne ka ek schedule banayein especially high-wattage appliances ke liye taaki overlap na ho.
  [0.409] (Monitoring) Smart plug ya energy monitor use karke pata karein kaunsa appliance sabse zyada bijli kharch kar raha hai.
  [0.397] (General) Solar water heater install karne se 

In [9]:
from datetime import datetime

def get_current_season():
    month = datetime.now().month
    if month in (3, 4, 5, 6):
        return "Seasonal-Summer"
    elif month in (11, 12, 1, 2):
        return "Seasonal-Winter"
    elif month == 10:
        return "Seasonal-Festival"   # Diwali season
    else:
        return None  # monsoon (Jul-Sep) - no special push


def merge_seasonal_tips(base_tips, energy_tips, season, extra=2):
    """Season-specific tips (jo already list mein nahi hain) ko front mein add karta hai."""
    if season is None:
        return base_tips
    existing_ids = {t["id"] for t in base_tips}
    seasonal_pool = [t for t in energy_tips if t["category"] == season and t["id"] not in existing_ids]
    boosted = []
    for t in seasonal_pool[:extra]:
        t = dict(t)
        t["relevance"] = "seasonal-boost"
        boosted.append(t)
    return boosted + base_tips


# ---- energy_agent ko update karo seasonal awareness ke saath ----
def energy_agent(usage_hours, k_tips=5):
    consumption = []
    for appliance, hours in usage_hours.items():
        watt = appliance_wattage.get(appliance, 0)
        monthly_kwh = (watt * hours * 30) / 1000
        consumption.append({"appliance": appliance, "hours_per_day": hours, "monthly_kwh": round(monthly_kwh, 1)})
    consumption.sort(key=lambda x: x["monthly_kwh"], reverse=True)

    total_units = round(sum(c["monthly_kwh"] for c in consumption), 1)
    bill = calculate_bill(total_units)
    proximity_advice = units_to_next_slab_saving(total_units)

    top_appliances = [c["appliance"] for c in consumption[:3]]
    query = "Energy saving tips for: " + ", ".join(top_appliances) if top_appliances else "general energy saving"
    tips = retrieve_tips(query, k=k_tips)

    # NEW: seasonal boost
    season = get_current_season()
    tips = merge_seasonal_tips(tips, energy_tips, season)

    return {
        "consumption_breakdown": consumption,
        "total_monthly_units": total_units,
        "bill": bill,
        "proximity_advice": proximity_advice,
        "recommended_tips": tips,
        "current_season": season,
    }


# ---- test ----
result = energy_agent(sample_usage)
print("Current season tag:", result["current_season"])
print("\nTips (seasonal-boosted ones pehle aayenge):")
for tip in result["recommended_tips"]:
    print(f"  [{tip['relevance']}] ({tip['category']}) {tip['tip']}")

Current season tag: Seasonal-Festival

Tips (seasonal-boosted ones pehle aayenge):
  [seasonal-boost] (Seasonal-Festival) Diwali/festival lights LED string lights use karein, traditional bulbs se 80% kam bijli kharch karte hain.
  [seasonal-boost] (Seasonal-Festival) Festival decoration lights ke liye timer laga dein, raat 11-12 baje ke baad automatically band ho jayein.
  [0.426] (Behavioral) Appliance use karne ka ek schedule banayein especially high-wattage appliances ke liye taaki overlap na ho.
  [0.409] (Monitoring) Smart plug ya energy monitor use karke pata karein kaunsa appliance sabse zyada bijli kharch kar raha hai.
  [0.397] (General) Solar water heater install karne se geyser ka 60-70% load khatam ho sakta hai, especially North India mein.
  [0.367] (General) Appliances ki regular servicing karayein, purane/dirty appliances zyada bijli kharch karte hain.
  [0.356] (Refrigerator) Fridge ko deewar se kam se kam 4-6 inch door rakhein taaki heat properly dissipate ho sake.


In [10]:
%%writefile app.py
import json
from datetime import datetime
import numpy as np
import streamlit as st
from sentence_transformers import SentenceTransformer
import faiss

st.set_page_config(page_title="Energy Optimizer Agent", page_icon="⚡", layout="centered")

PUNJAB_SLABS = [
    (100, 3.85),
    (300, 5.25),
    (500, 6.95),
    (float("inf"), 8.15),
]
FIXED_CHARGE = 130
ELECTRICITY_DUTY = 0.05
FREE_UNITS = 300


def calculate_slab_bill(units):
    if units <= 0:
        return 0.0
    remaining, prev_limit, total = units, 0, 0.0
    for limit, rate in PUNJAB_SLABS:
        slab_units = min(remaining, limit - prev_limit)
        if slab_units <= 0:
            break
        total += slab_units * rate
        remaining -= slab_units
        prev_limit = limit
        if remaining <= 0:
            break
    return total


def calculate_bill(units):
    if units <= FREE_UNITS:
        return {
            "units": units, "free_scheme_applied": True,
            "energy_charge": 0.0, "fixed_charge": 0.0, "duty": 0.0, "total": 0.0,
        }
    energy_charge = calculate_slab_bill(units)
    duty = energy_charge * ELECTRICITY_DUTY
    total = energy_charge + FIXED_CHARGE + duty
    return {
        "units": units, "free_scheme_applied": False,
        "energy_charge": round(energy_charge, 2), "fixed_charge": FIXED_CHARGE,
        "duty": round(duty, 2), "total": round(total, 2),
    }


def units_to_next_slab_saving(units):
    if units <= FREE_UNITS:
        return None
    over = units - FREE_UNITS
    return {
        "units_over_free_limit": round(over, 1),
        "advice": (f"Aap free {FREE_UNITS}-unit limit se {over:.1f} units zyada use kar rahe ho. "
                   f"Agar is mahine {over:.1f} units kam kar lein, poora bill Rs 0 ho sakta hai "
                   f"(sirf extra units nahi, poora bill — Punjab ke cliff rule ki wajah se)."),
    }


@st.cache_resource(show_spinner="Loading model and knowledge base...")
def load_resources():
    with open("data/appliance_wattage.json") as f:
        appliance_wattage = json.load(f)
    with open("data/energy_tips.json") as f:
        energy_tips = json.load(f)

    model = SentenceTransformer("all-MiniLM-L6-v2")
    tip_texts = [t["tip"] for t in energy_tips]
    tip_embeddings = model.encode(tip_texts, normalize_embeddings=True).astype("float32")

    dimension = tip_embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(tip_embeddings)

    return appliance_wattage, energy_tips, model, index


appliance_wattage, energy_tips, model, index = load_resources()


def retrieve_tips(query, k=5):
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, k)
    return [{**energy_tips[i], "relevance": round(float(s), 3)} for s, i in zip(scores[0], idx[0])]


def get_current_season():
    month = datetime.now().month
    if month in (3, 4, 5, 6):
        return "Seasonal-Summer"
    elif month in (11, 12, 1, 2):
        return "Seasonal-Winter"
    elif month == 10:
        return "Seasonal-Festival"
    else:
        return None


def merge_seasonal_tips(base_tips, all_tips, season, extra=2):
    if season is None:
        return base_tips
    existing_ids = {t["id"] for t in base_tips}
    seasonal_pool = [t for t in all_tips if t["category"] == season and t["id"] not in existing_ids]
    boosted = []
    for t in seasonal_pool[:extra]:
        t = dict(t)
        t["relevance"] = "seasonal-boost"
        boosted.append(t)
    return boosted + base_tips


def energy_agent(usage_hours, k_tips=5):
    consumption = []
    for appliance, hours in usage_hours.items():
        watt = appliance_wattage.get(appliance, 0)
        monthly_kwh = (watt * hours * 30) / 1000
        consumption.append({"appliance": appliance, "hours_per_day": hours, "monthly_kwh": round(monthly_kwh, 1)})
    consumption.sort(key=lambda x: x["monthly_kwh"], reverse=True)

    total_units = round(sum(c["monthly_kwh"] for c in consumption), 1)
    bill = calculate_bill(total_units)
    proximity_advice = units_to_next_slab_saving(total_units)

    top_appliances = [c["appliance"] for c in consumption[:3]]
    query = "Energy saving tips for: " + ", ".join(top_appliances) if top_appliances else "general energy saving"
    tips = retrieve_tips(query, k=k_tips)

    season = get_current_season()
    tips = merge_seasonal_tips(tips, energy_tips, season)

    return {
        "consumption_breakdown": consumption,
        "total_monthly_units": total_units,
        "bill": bill,
        "proximity_advice": proximity_advice,
        "recommended_tips": tips,
        "current_season": season,
    }


st.title("⚡ Energy Consumption Optimizer Agent")
st.caption("India-specific (Punjab/PSPCL) · RAG-powered tips · Built for Build Next 2026")

st.markdown(
    "Apne ghar ke appliances aur unka daily usage daalo — ye AI agent tumhara "
    "estimated bill nikalega aur personalized energy-saving tips dega."
)

st.subheader("Apna usage daalo")

selected_appliances = st.multiselect(
    "Appliances select karo jo tum roz use karte ho",
    options=list(appliance_wattage.keys()),
    default=["Air Conditioner (1.5 Ton, 3 Star)", "Refrigerator (Double Door, 250L)"],
)

usage_hours = {}
if selected_appliances:
    st.write("Har appliance ke liye roz kitne ghante use hota hai:")
    cols = st.columns(2)
    for i, appliance in enumerate(selected_appliances):
        with cols[i % 2]:
            hours = st.number_input(
                appliance, min_value=0.0, max_value=24.0, value=4.0, step=0.5,
                key=f"hrs_{appliance}",
            )
            usage_hours[appliance] = hours

if st.button("🔍 Check my bill & get tips", type="primary", disabled=not selected_appliances):
    result = energy_agent(usage_hours)

    st.subheader("📊 Consumption Breakdown")
    for c in result["consumption_breakdown"]:
        st.write(f"**{c['appliance']}** — {c['hours_per_day']} hrs/day → {c['monthly_kwh']} kWh/month")

    st.metric("Total Estimated Monthly Units", f"{result['total_monthly_units']} kWh")

    st.subheader("💰 Estimated Bill (PSPCL, Punjab)")
    bill = result["bill"]
    if bill["free_scheme_applied"]:
        st.success(f"Bill: ₹0 — within the {FREE_UNITS} free units/month scheme! 🎉")
    else:
        st.error(f"Total Bill: ₹{bill['total']}")
        st.write(f"Energy charge: ₹{bill['energy_charge']} | Fixed charge: ₹{bill['fixed_charge']} | Duty: ₹{bill['duty']}")

    if result["proximity_advice"]:
        st.warning(result["proximity_advice"]["advice"])

    st.subheader("💡 Personalized Energy-Saving Tips")
    if result["current_season"]:
        st.caption(f"🗓️ Season-aware: {result['current_season'].replace('Seasonal-', '')} tips boosted for this time of year")
    for tip in result["recommended_tips"]:
        badge = "🌟 Seasonal" if tip["relevance"] == "seasonal-boost" else f"relevance: {tip['relevance']}"
        st.info(f"**[{tip['category']}]** {tip['tip']}  \n*({badge})*")

st.markdown("---")
st.caption(
    "⚠️ Billing rates are approximate (PSERC FY 2026-27 order) and for demonstration purposes. "
    "Verify against your actual PSPCL bill for exact figures."
)

Overwriting app.py


In [7]:
%%writefile requirements.txt
streamlit
sentence-transformers
faiss-cpu
numpy

Writing requirements.txt


In [8]:
import os, json
os.makedirs("data", exist_ok=True)

with open("data/appliance_wattage.json", "w") as f:
    json.dump(appliance_wattage, f, indent=2)
with open("data/energy_tips.json", "w") as f:
    json.dump(energy_tips, f, indent=2)

print("Files ready:", os.listdir("data"))

Files ready: ['appliance_wattage.json', 'energy_tips.json']
